<a href="https://colab.research.google.com/github/Likith-Reddy25/Summer-Intern/blob/main/codes/MNIST_1D_PCA_4_Covariant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Covariant QKE


In [ ]:
!pip install qiskit qiskit_machine_learning qiskit-algorithms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.8 MB/s eta 0:00:00


In [ ]:
!pip install mnist1d --break-system-packages

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
import mnist1d

# ----------------------------- Custom Covariant Map -----------------------
def CovariantFeatureMap(feature_dimension):
    """
    Constructs a group-covariant feature map.
    Applies a data-dependent unitary representation of the translation
    group U(x) = exp(-i * x * Z) to a fixed, entangled fiducial state.
    """
    x = ParameterVector('x', length=feature_dimension)
    qc = QuantumCircuit(feature_dimension, name="CovariantMap")

    # 1. Prepare the fiducial state (highly entangled starting state)
    for i in range(feature_dimension):
        qc.h(i)
    if feature_dimension > 1:
        for i in range(feature_dimension - 1):
            qc.cx(i, i + 1)

    # 2. Covariant data encoding layer
    # Note: A strictly covariant map for continuous data applies the
    # representation once, avoiding interleaved entanglement that breaks symmetry.
    for i in range(feature_dimension):
        qc.rz(x[i], i)

    return qc

# ----------------------------- Config ------------------------------------
N_DIM = 4
N_TRAIN = 250
N_TEST = 250
N_REPS = 30
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
CV_FOLDS = 5
DIGITS = (3, 5)

FEATURE_MAP = CovariantFeatureMap(feature_dimension=N_DIM)
KERNEL = FidelityStatevectorKernel(feature_map=FEATURE_MAP)


# ------------------------- Data loading -----------------------------------
def load_mnist1d_binary(digits=DIGITS):
    """Load MNIST-1D and keep only the two requested classes."""
    args = mnist1d.data.get_dataset_args()
    data = mnist1d.data.make_dataset(args)
    X = np.concatenate([data["x"], data["x_test"]])
    y = np.concatenate([data["y"], data["y_test"]])
    mask = np.isin(y, digits)
    X, y = X[mask], y[mask]
    y = np.where(y == digits[0], -1, 1)
    return X, y


def scaled_kernel(X1, X2, lam):
    """Fidelity kernel matrix with angle vectors pre-scaled by bandwidth lam."""
    return KERNEL.evaluate(x_vec=X1 * lam, y_vec=X2 * lam)


# ------------------------- Experiment loop ---------------------------------
def run_experiment(X_raw, y, n_reps=N_REPS, verbose=True):
    results = {
        "train": {"acc": [], "kappa": [], "f1": []},
        "test": {"acc": [], "kappa": [], "f1": []},
    }

    for rep in range(n_reps):
        X_tr_full, X_te_full, y_tr, y_te = train_test_split(
            X_raw, y,
            train_size=N_TRAIN, test_size=N_TEST,
            stratify=y, random_state=rep,
        )

        # PCA fit on train only, applied to both splits
        pca = PCA(n_components=N_DIM, random_state=rep)
        X_tr_pca = pca.fit_transform(X_tr_full)
        X_te_pca = pca.transform(X_te_full)

        # Angle-encoding scale: [0, 2*pi], parameters learned from train only
        scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
        X_tr = scaler.fit_transform(X_tr_pca)
        X_te = scaler.transform(X_te_pca)

        # ---- Grid search over (lambda, C) via 5-fold CV on train ----
        best_score, best_C, best_lam = -1.0, None, None
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=rep)

        for lam in LAMBDA_GRID:
            K_full = scaled_kernel(X_tr, X_tr, lam)
            for C in C_GRID:
                fold_acc = []
                for tr_idx, val_idx in skf.split(X_tr, y_tr):
                    K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                    K_val = K_full[np.ix_(val_idx, tr_idx)]
                    clf = SVC(kernel="precomputed", C=C)
                    clf.fit(K_tr, y_tr[tr_idx])
                    pred = clf.predict(K_val)
                    fold_acc.append(accuracy_score(y_tr[val_idx], pred))
                mean_acc = float(np.mean(fold_acc))
                if mean_acc > best_score:
                    best_score, best_C, best_lam = mean_acc, C, lam

        # ---- Refit on full train split with the winning hyperparams ----
        K_train = scaled_kernel(X_tr, X_tr, best_lam)
        K_test = scaled_kernel(X_te, X_tr, best_lam)

        clf = SVC(kernel="precomputed", C=best_C)
        clf.fit(K_train, y_tr)

        pred_train = clf.predict(K_train)
        pred_test = clf.predict(K_test)

        results["train"]["acc"].append(accuracy_score(y_tr, pred_train))
        results["train"]["kappa"].append(cohen_kappa_score(y_tr, pred_train))
        results["train"]["f1"].append(f1_score(y_tr, pred_train, average="macro"))

        results["test"]["acc"].append(accuracy_score(y_te, pred_test))
        results["test"]["kappa"].append(cohen_kappa_score(y_te, pred_test))
        results["test"]["f1"].append(f1_score(y_te, pred_test, average="macro"))

        if verbose:
            print(f"[rep {rep + 1:2d}/{n_reps}] best_C={best_C:<6} "
                  f"best_lambda={best_lam:<6} "
                  f"test_acc={results['test']['acc'][-1]:.3f}")

    return results


def fmt(vals):
    return f"{np.mean(vals):.3f} ({np.std(vals):.3f})"


def print_table2_row(results, dataset="MNIST-1D-PCA-4", mapping="CovariantMap"):
    print(f"\n=== Table 2 style summary: {dataset} | qSVM_Cov (plain QKE, n={len(results['test']['acc'])}) ===")
    header = f"{'Dataset':<18}{'Map':<16}{'Params':<10}{'Metric':<16}{'Train':<18}{'Test':<18}"
    print(header)
    print("-" * len(header))
    for metric, label in [("acc", "Accuracy"), ("kappa", "Cohen's kappa"), ("f1", "Macro F1")]:
        print(f"{dataset:<18}{mapping:<16}{'-':<10}{label:<16}"
              f"{fmt(results['train'][metric]):<18}{fmt(results['test'][metric]):<18}")


if __name__ == "__main__":
    X_raw, y = load_mnist1d_binary()
    print(f"Loaded MNIST-1D digits {DIGITS}: {X_raw.shape[0]} samples, "
          f"raw dim={X_raw.shape[1]}")

    results = run_experiment(X_raw, y, n_reps=N_REPS)
    print_table2_row(results)

Loaded MNIST-1D digits (3, 5): 1000 samples, raw dim=40
[rep  1/30] best_C=10     best_lambda=0.5    test_acc=0.920
[rep  2/30] best_C=1      best_lambda=0.5    test_acc=0.940
[rep  3/30] best_C=1      best_lambda=0.5    test_acc=0.844
[rep  4/30] best_C=100    best_lambda=0.1    test_acc=0.868
[rep  5/30] best_C=10     best_lambda=0.5    test_acc=0.888
[rep  6/30] best_C=1      best_lambda=0.5    test_acc=0.880
[rep  7/30] best_C=10     best_lambda=0.5    test_acc=0.900
[rep  8/30] best_C=1      best_lambda=0.5    test_acc=0.868
[rep  9/30] best_C=1      best_lambda=0.5    test_acc=0.896
[rep 10/30] best_C=1      best_lambda=0.5    test_acc=0.880
[rep 11/30] best_C=1      best_lambda=0.5    test_acc=0.876
[rep 12/30] best_C=0.1    best_lambda=0.5    test_acc=0.908
[rep 13/30] best_C=10     best_lambda=0.5    test_acc=0.824
[rep 14/30] best_C=0.1    best_lambda=0.5    test_acc=0.904
[rep 15/30] best_C=1      best_lambda=0.5    test_acc=0.856
[rep 16/30] best_C=1      best_lambda=1.0   

Covariant Shared 3

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
import mnist1d

# ----------------------------- Config ------------------------------------
N_DIM = 4
N_TRAIN = 250
N_TEST = 250
N_REPS = 30
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
CV_FOLDS = 5
DIGITS = (3, 5)

# --- QKT Settings ---
# Modified to run on the full training set to closer match the paper's
# rigorous evaluation (Warning: simulation time will be significantly higher)
QKT_MAXITER = 100

# ----------------------------- Feature Map --------------------------------
def CovariantFeatureMap_Shared3(feature_dimension):
    """
    Constructs a group-covariant feature map with a parameterized
    fiducial state using a 'Shared (3)' strategy.
    """
    x = ParameterVector('x', length=feature_dimension)
    theta = ParameterVector('θ', length=3)
    qc = QuantumCircuit(feature_dimension, name="CovariantMap_QKT_Shared3")

    # 1. Parameterized Fiducial State
    for layer in range(3):
        for i in range(feature_dimension):
            qc.ry(theta[layer], i)
        if layer < 2 and feature_dimension > 1:
            for i in range(feature_dimension - 1):
                qc.cx(i, i + 1)

    # 2. Covariant Data Encoding Layer (U(x) = exp(-i * x * Z))
    for i in range(feature_dimension):
        qc.rz(x[i], i)

    return qc, theta, x

PARAM_CIRCUIT, WEIGHT_PARAMS, DATA_PARAMS = CovariantFeatureMap_Shared3(feature_dimension=N_DIM)

# ------------------------- Helper Functions -------------------------------
def load_mnist1d_binary(digits=DIGITS):
    args = mnist1d.data.get_dataset_args()
    data = mnist1d.data.make_dataset(args)
    X = np.concatenate([data["x"], data["x_test"]])
    y = np.concatenate([data["y"], data["y_test"]])
    mask = np.isin(y, digits)
    X, y = X[mask], y[mask]
    y = np.where(y == digits[0], -1, 1)
    return X, y

def get_bound_kernel(theta_vals):
    param_dict = dict(zip(WEIGHT_PARAMS, theta_vals))
    bound_circuit = PARAM_CIRCUIT.assign_parameters(param_dict)
    return FidelityStatevectorKernel(feature_map=bound_circuit)

def centered_kta(K, y):
    n = len(y)
    H = np.eye(n) - np.ones((n, n)) / n
    K_c = H @ K @ H
    Y = np.outer(y, y)
    Y_c = H @ Y @ H

    inner = np.sum(K_c * Y_c)
    norm_K = np.linalg.norm(K_c, 'fro')
    norm_Y = np.linalg.norm(Y_c, 'fro')

    if norm_K == 0 or norm_Y == 0:
        return 0.0
    return inner / (norm_K * norm_Y)

# ------------------------- Experiment loop ---------------------------------
def run_experiment(X_raw, y, n_reps=N_REPS, verbose=True):
    results = {
        "train": {"acc": [], "kappa": [], "f1": []},
        "test": {"acc": [], "kappa": [], "f1": []},
    }

    for rep in range(n_reps):
        X_tr_full, X_te_full, y_tr, y_te = train_test_split(
            X_raw, y,
            train_size=N_TRAIN, test_size=N_TEST,
            stratify=y, random_state=rep,
        )

        pca = PCA(n_components=N_DIM, random_state=rep)
        X_tr_pca = pca.fit_transform(X_tr_full)
        X_te_pca = pca.transform(X_te_full)

        scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
        X_tr = scaler.fit_transform(X_tr_pca)
        X_te = scaler.transform(X_te_pca)

        # ---- PHASE 1: QKT Optimization (Centered KTA) ----
        # Using full X_tr instead of a 50-sample subset
        def qkt_objective(theta):
            kernel = get_bound_kernel(theta)
            K_train = kernel.evaluate(x_vec=X_tr)
            return -centered_kta(K_train, y_tr)

        initial_theta = np.random.uniform(-np.pi, np.pi, size=3)

        res = minimize(qkt_objective, initial_theta, method='COBYLA', options={'maxiter': QKT_MAXITER})
        best_theta = res.x

        opt_kernel = get_bound_kernel(best_theta)

        def scaled_kernel_opt(X1, X2, lam):
            return opt_kernel.evaluate(x_vec=X1 * lam, y_vec=X2 * lam)

        # ---- PHASE 2: Grid search over (lambda, C) via CV on train ----
        best_score, best_C, best_lam = -1.0, None, None
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=rep)

        for lam in LAMBDA_GRID:
            K_full = scaled_kernel_opt(X_tr, X_tr, lam)
            for C in C_GRID:
                fold_acc = []
                for tr_idx, val_idx in skf.split(X_tr, y_tr):
                    K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                    K_val = K_full[np.ix_(val_idx, tr_idx)]
                    clf = SVC(kernel="precomputed", C=C)
                    clf.fit(K_tr, y_tr[tr_idx])
                    pred = clf.predict(K_val)
                    fold_acc.append(accuracy_score(y_tr[val_idx], pred))
                mean_acc = float(np.mean(fold_acc))
                if mean_acc > best_score:
                    best_score, best_C, best_lam = mean_acc, C, lam

        # ---- PHASE 3: Refit on full train split with winning hyperparams ----
        K_train = scaled_kernel_opt(X_tr, X_tr, best_lam)
        K_test = scaled_kernel_opt(X_te, X_tr, best_lam)

        clf = SVC(kernel="precomputed", C=best_C)
        clf.fit(K_train, y_tr)

        pred_train = clf.predict(K_train)
        pred_test = clf.predict(K_test)

        results["train"]["acc"].append(accuracy_score(y_tr, pred_train))
        results["train"]["kappa"].append(cohen_kappa_score(y_tr, pred_train))
        results["train"]["f1"].append(f1_score(y_tr, pred_train, average="macro"))

        results["test"]["acc"].append(accuracy_score(y_te, pred_test))
        results["test"]["kappa"].append(cohen_kappa_score(y_te, pred_test))
        results["test"]["f1"].append(f1_score(y_te, pred_test, average="macro"))

        if verbose:
            print(f"[rep {rep + 1:2d}/{n_reps}] KTA={-res.fun:.3f} best_C={best_C:<6} "
                  f"best_lambda={best_lam:<6} test_acc={results['test']['acc'][-1]:.3f}")

    return results

def fmt(vals):
    return f"{np.mean(vals):.3f} ({np.std(vals):.3f})"

def print_table2_row(results, dataset="MNIST-1D-PCA-4", mapping="CovariantMap-Shared"):
    print(f"\n=== Table 2 style summary: {dataset} | Covariant QKT (n={len(results['test']['acc'])}) ===")
    header = f"{'Dataset':<18}{'Map':<24}{'Params':<10}{'Metric':<16}{'Train':<18}{'Test':<18}"
    print(header)
    print("-" * len(header))
    for metric, label in [("acc", "Accuracy"), ("kappa", "Cohen's kappa"), ("f1", "Macro F1")]:
        print(f"{dataset:<18}{mapping:<24}{'3':<10}{label:<16}"
              f"{fmt(results['train'][metric]):<18}{fmt(results['test'][metric]):<18}")

if __name__ == "__main__":
    X_raw, y = load_mnist1d_binary()
    print(f"Loaded MNIST-1D digits {DIGITS}: {X_raw.shape[0]} samples, raw dim={X_raw.shape[1]}")
    results = run_experiment(X_raw, y, n_reps=N_REPS)
    print_table2_row(results)

Loaded MNIST-1D digits (3, 5): 1000 samples, raw dim=40
[rep  1/30] KTA=0.432 best_C=1      best_lambda=1.0    test_acc=0.900
[rep  2/30] KTA=0.368 best_C=10     best_lambda=0.5    test_acc=0.944
[rep  3/30] KTA=0.272 best_C=10     best_lambda=0.5    test_acc=0.844
[rep  4/30] KTA=0.327 best_C=0.1    best_lambda=1.0    test_acc=0.868
[rep  5/30] KTA=0.385 best_C=100    best_lambda=0.5    test_acc=0.888
[rep  6/30] KTA=0.262 best_C=10     best_lambda=0.5    test_acc=0.892
[rep  7/30] KTA=0.343 best_C=100    best_lambda=0.5    test_acc=0.892
[rep  8/30] KTA=0.355 best_C=1      best_lambda=0.5    test_acc=0.872
[rep  9/30] KTA=0.330 best_C=10     best_lambda=0.5    test_acc=0.872
[rep 10/30] KTA=0.413 best_C=10     best_lambda=0.5    test_acc=0.896
[rep 11/30] KTA=0.311 best_C=100    best_lambda=1.0    test_acc=0.852
[rep 12/30] KTA=0.315 best_C=1      best_lambda=0.5    test_acc=0.900
[rep 13/30] KTA=0.379 best_C=10     best_lambda=0.5    test_acc=0.840
[rep 14/30] KTA=0.456 best_C=10   